# ViT Memory Autoencoder Training (Colab)

This notebook runs the rebuilt module-based pipeline using `train.py` and `evaluate.py`.

In [1]:
# Install dependencies
!pip install -q -r requirements.txt
print('Dependencies installed')

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Dependencies installed


In [ ]:
import os
import json

# Kaggle credentials
KAGGLE_USERNAME = "eftekin"
KAGGLE_KEY = ""   # empty — user fills this in

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
cred_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(cred_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(cred_path, 0o600)
print('Kaggle credentials saved to ~/.kaggle/kaggle.json')

Kaggle credentials saved to ~/.kaggle/kaggle.json


In [3]:
import os
import shutil
import subprocess

project_root = '/content/explainable-anomaly-detection'
if os.path.isdir(project_root):
    os.chdir(project_root)

mvtec_root = os.path.join('data', 'mvtec')
categories = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

if not os.path.exists(mvtec_root):
    os.makedirs(mvtec_root, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'ipythonx/mvtec-ad',
        '-p', '/content',
        '--unzip'
    ], check=True)
    for cat in categories:
        src = os.path.join('/content', cat)
        dst = os.path.join(mvtec_root, cat)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.move(src, dst)

print('Dataset root:', mvtec_root)

Dataset root: data/mvtec


In [4]:
# Run training
!python train.py --data-root data/mvtec --category bottle

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [5]:
# Run evaluation
!python evaluate.py --data-root data/mvtec --category bottle --checkpoint checkpoints/best_model.pth

python3: can't open file '/content/evaluate.py': [Errno 2] No such file or directory


In [6]:
!pip install transformers scikit-learn tqdm kaggle -q
!git clone https://github.com/eftekin/explainable-anomaly-detection
%cd explainable-anomaly-detection

Cloning into 'explainable-anomaly-detection'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 155 (delta 59), reused 138 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 2.23 MiB | 6.18 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/explainable-anomaly-detection


In [ ]:
import os, shutil

# Kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'eftekin'
os.environ['KAGGLE_KEY'] = ''

# Download MVTec AD (ipythonx/mvtec-ad — 5.2 GB, extracts category folders directly)
!kaggle datasets download -d ipythonx/mvtec-ad -p ./data --unzip -q

# Show all extracted top-level folders
print("Extracted folders:", os.listdir('./data'))

# Fix structure: data/bottle -> data/mvtec/bottle
os.makedirs('data/mvtec', exist_ok=True)
if os.path.exists('data/bottle'):
    shutil.move('data/bottle', 'data/mvtec/bottle')
    print("Moved data/bottle -> data/mvtec/bottle")

print("\ndata/mvtec/bottle/:",      os.listdir('data/mvtec/bottle'))
print("data/mvtec/bottle/train/:",  os.listdir('data/mvtec/bottle/train'))
print("data/mvtec/bottle/test/:",   os.listdir('data/mvtec/bottle/test'))


Dataset URL: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
License(s): copyright-authors
Extracted folders: ['cable', 'carpet', 'zipper', 'hazelnut', 'leather', 'readme.txt', 'license.txt', 'bottle', 'capsule', 'pill', 'transistor', 'grid', 'metal_nut', 'toothbrush', 'wood', 'screw', 'tile']
Moved data/bottle -> data/mvtec/bottle

data/mvtec/bottle/: ['ground_truth', 'train', 'readme.txt', 'license.txt', 'test']
data/mvtec/bottle/train/: ['good']
data/mvtec/bottle/test/: ['broken_small', 'broken_large', 'contamination', 'good']


In [8]:
!python train.py

Device: cuda
Config | category=bottle | image=384 | tau=0.05 | lambda_ent=0.1 | freeze=50
model.safetensors: 100% 347M/347M [00:02<00:00, 123MB/s]  
Training 100 epochs | tau=0.05 | lambda_ent=0.1 | freeze_until=50
ep 001/100 | total=0.49933 | recon=0.12825 | ent=0.37108 | phase=1
Traceback (most recent call last):            
  File "/usr/lib/python3.12/threading.py", line 359, in wait
    gotit = waiter.acquire(True, timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/queue.py", line 180, in get
    self.not_empty.wait(remaining)
  File "/usr/lib/python3.12/threading.py", line 364, in wait
    self._acquire_restore(saved_state)
  File "/usr/lib/python3.12/threading.py", line 311, in _acquire_restore
    def _acquire_restore(self, x):

KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most

In [ ]:
!python evaluate.py --checkpoint checkpoints/best_model.pth